# 04 · Desafio adicional: Metadados e Exploração Contextual

**Tech Challenge – Fase 4 · Voice of Customer Intelligence com RAG** · RM371084

A busca semântica encontra avaliações **sobre** um tema, mas não sabe **de quem** ou **de quando** elas são.
Perguntas como *“problemas nas avaliações com nota baixa”* ou *“elogios em cama, mesa e banho”* dependem de
atributos que **não estão no texto** — estão nos metadados do pedido.

Como todos os metadados já foram armazenados junto aos vetores no ChromaDB (notebook 01), o pipeline foi estendido:

1. Na **transformação da pergunta**, o Gemini também extrai **filtros explícitos** (nota, categoria, período, atraso
   na entrega, UF) — somente quando a pergunta os menciona;
2. Os filtros são **validados** contra os valores existentes na base (categorias e UFs inválidas são descartadas,
   notas são limitadas a 1–5) e convertidos em um filtro `where` do ChromaDB;
3. A busca vetorial é feita **apenas dentro do subconjunto filtrado** (pré-filtragem), e o perfil do tema e as
   evidências refletem esse recorte.

| Metadado | Filtro | Exemplo de pergunta |
|---|---|---|
| `review_score` | faixa de nota | “…nas avaliações com nota baixa?” |
| `product_category` | categorias | “…em produtos de informática?” |
| `year_month` | período (lista de meses) | “…em maio e junho de 2018?” |
| `delivered_late` | entregue após o prazo | “…em entregas atrasadas?” |
| `customer_state` | UF do cliente | “…de clientes do Rio de Janeiro?” |

In [1]:
import sys
sys.path.insert(0, "../src")
import pandas as pd
from vocrag import config, llm
from vocrag.filters import build_where, CATEGORIES
from vocrag.rag import VoCRAG
from vocrag.display import show

rag = VoCRAG()
print(f"{len(CATEGORIES)} categorias disponíveis, ex.: {CATEGORIES[:8]}")

74 categorias disponíveis, ex.: ['agro_industria_e_comercio', 'alimentos', 'alimentos_bebidas', 'artes', 'artes_e_artesanato', 'artigos_de_festas', 'artigos_de_natal', 'audio']


## 1. Extração de filtros a partir da pergunta

A extração acontece na mesma chamada que já classificava o escopo e reformulava a pergunta (nenhuma chamada extra
ao LLM). O resultado validado vira um filtro do ChromaDB:

In [2]:
for q in ["Quais são os principais problemas relatados nas avaliações com nota baixa?",
          "O que os clientes mais elogiam em produtos de cama, mesa e banho?",
          "Do que os clientes reclamavam em maio e junho de 2018?",
          "O que os clientes relatam sobre a entrega?"]:
    a = llm.analyze_question(q)
    print(q); print("  filtros extraídos:", a.filters.model_dump(exclude_none=True)); print("  where:", build_where(a.filters)); print()

Quais são os principais problemas relatados nas avaliações com nota baixa?
  filtros extraídos: {'max_score': 2}
  where: {'review_score': {'$lte': 2}}



O que os clientes mais elogiam em produtos de cama, mesa e banho?
  filtros extraídos: {'min_score': 4, 'product_categories': ['cama_mesa_banho']}
  where: {'$and': [{'review_score': {'$gte': 4}}, {'product_category': {'$in': ['cama_mesa_banho']}}]}



Do que os clientes reclamavam em maio e junho de 2018?
  filtros extraídos: {'max_score': 2, 'month_from': '2018-05', 'month_to': '2018-06'}
  where: {'$and': [{'review_score': {'$lte': 2}}, {'year_month': {'$in': ['2018-05', '2018-06']}}]}



O que os clientes relatam sobre a entrega?
  filtros extraídos: {}
  where: None



## 2. Perguntas contextualizadas

### Problemas nas avaliações com nota baixa

In [3]:
r1 = rag.ask("Quais são os principais problemas relatados nas avaliações com nota baixa?"); show(r1)

### ❓ Quais são os principais problemas relatados nas avaliações com nota baixa?

**Status:** ✅ Respondida com evidências

**Filtro de metadados aplicado:** `{'review_score': {'$lte': 2}}`

**Consultas usadas na busca:** `Quais são os principais problemas relatados nas avaliações com nota baixa?` · `o produto veio com defeito e fiquei muito decepcionado` · `a entrega atrasou demais e o atendimento foi péssimo` · `veio faltando peças e o produto não funciona direito`

**Perfil do tema:** ≥1971 avaliações relacionadas · nota média 1.18 · 100.0% com nota 1-2

**Modelo de geração:** `gemini-3.5-flash-lite`

---

Com base nas avaliações recuperadas, os principais problemas relatados pelos clientes com notas baixas podem ser agrupados nos seguintes temas:

- **Problemas com o produto (defeitos e itens faltando/incompletos):** Vários clientes relataram receber produtos com defeitos de fabricação, itens danificados ou com mau funcionamento [R5, R9, R11, R14, R15]. Também há relatos recorrentes de produtos que vieram incompletos, com peças faltando em relação ao que foi comprado ou descrito na nota fiscal [R1, R8, R9, R12, R13]. Além disso, foram relatadas más condições de entrega do produto, como embalagens sujas [R4].

- **Atrasos na entrega e falhas logísticas:** Há diversas reclamações sobre demora excessiva, atrasos expressivos nas entregas e pedidos não entregues ou cancelados [R2, R3, R4, R6, R10]. Clientes também apontam a falta de aviso ou satisfação por parte da empresa sobre o status do envio [R6, R10].

- **Péssimo atendimento e dificuldade de suporte:** Clientes relataram atendimento ruim e ausência de retorno ao tentarem contato para resolver problemas de entrega, solicitar estornos ou realizar trocas de produtos defeituosos [R3, R7, R11, R12].


> **Limitações:** A análise baseia-se em uma amostra recuperada de 15 avaliações dentro de um universo estimado de 1971 avaliações com nota baixa (nota <= 2), onde a nota média é 1.18 e 27.1% dos pedidos entregues apresentam atraso. O perfil de categorias mais frequentes inclui beleza_saude, cama_mesa_banho, relogios_presentes, informatica_acessorios e moveis_decoracao, embora a amostra textual detalhada cubra apenas uma fração restrita desses setores.

**Evidências citadas (15):**

ref,similaridade,nota,categoria,data,atraso_dias,texto_identico_x,avaliacao
R1,0.933000,1,beleza_saude,2017-09,-12.000000,1,O produto veio com peças faltando. Não consigo usá-lo.
R2,0.927800,1,cama_mesa_banho,2018-02,nan,1,Pessimo muita demora na entrega
R3,0.930000,1,eletrodomesticos,2018-02,nan,1,"Entrega nao foi feita , pedsimo atendimento . E nao fazem estorno"
R4,0.924100,1,esporte_lazer,2018-08,1.000000,1,"Decepcionada com a compra. O produto foi entregue com atraso, o pior não foi isso, o mesmo chegou solto em uma caixa suja, tive que usar ""panos"" para limpar a embalagem antes de abrir. A sujeira me deixou indignada!"
R5,0.932700,1,relogios_presentes,2018-07,-14.000000,3,Produto veio com defeito
R6,0.930300,1,utilidades_domesticas,2018-07,33.000000,1,ATRASO ENORME NA ENTREGA. A entrega está muuuuito atrasada e não recebi nenhum retorno a respeito.
R7,0.926800,1,perfumaria,2018-05,49.000000,1,Entrega e Atendimento. Péssimo atendimento!
R8,0.919700,1,bebes,2018-01,-27.000000,1,Falta parte do produto
R9,0.924300,1,eletronicos,2017-08,-18.000000,1,"Efetuei a compra de dois fones desse modelo mesmo, porém só veio um e o que veio está com defeito. Desejo a devolução e me declaro completamente insatisfeita."
R10,0.924100,2,cama_mesa_banho,2018-06,20.000000,1,péssimo *. mercadoria super atrasada e nem satisfação


### Elogios em uma categoria de produtos

In [4]:
r2 = rag.ask("O que os clientes mais elogiam em produtos de cama, mesa e banho?"); show(r2)

### ❓ O que os clientes mais elogiam em produtos de cama, mesa e banho?

**Status:** ✅ Respondida com evidências

**Filtro de metadados aplicado:** `{'$and': [{'review_score': {'$gte': 4}}, {'product_category': {'$in': ['cama_mesa_banho']}}]}`

**Consultas usadas na busca:** `O que os clientes mais elogiam em produtos de cama, mesa e banho?` · `o tecido é de ótima qualidade e muito macio` · `fiquei muito satisfeito com os lençóis e toalhas` · `excelente produto, superou minhas expectativas` · `chegou no prazo e o material é maravilhoso`

**Perfil do tema:** 890 avaliações relacionadas · nota média 4.83 · 0.0% com nota 1-2

**Modelo de geração:** `gemini-3.5-flash-lite`

---

Com base na amostra recuperada de 890 avaliações (nota média de 4.83) na categoria de cama, mesa e banho, os clientes destacam os seguintes pontos de elogio:

- **Qualidade e superação de expectativas**: Os clientes elogiam a excelente qualidade dos produtos, mencionando que superaram suas expectativas [R1, R6], que o material é muito bom [R8, R9, R10], e citando características específicas como tecido macio e confortável [R7], tapetes lindos [R5], toalhas boas [R4], manta boa [R15] e travesseiro com regulagem [R14]. Um cliente apontou que o produto é muito transparente, mas ainda assim considerou o material muito bom [R10].

- **Agilidade e cumprimento do prazo de entrega**: Há forte recorrência de elogios à entrega, com muitos pedidos chegando antes do prazo previsto [R3, R5, R9, R10, R11] ou exatamente no prazo [R2, R8, R12, R15]. Um cliente relatou extrema satisfação após receber o produto no dia seguinte à compra [R14]. O perfil geral indica que apenas 2,5% dos pedidos da amostra foram entregues após o prazo.

- **Embalagem adequada**: A qualidade da embalagem também foi mencionada positivamente entre as entregas no prazo [R2].


> **Limitações:** A análise baseia-se em uma amostra de 890 avaliações recuperadas no recorte de cama, mesa e banho com notas maiores ou iguais a 4, o que representa um viés positivo (0.0% com nota 1 ou 2 na amostra relacionada), limitando a visibilidade sobre problemas críticos, embora traga forte evidência sobre os motivadores de satisfação.

**Evidências citadas (14):**

ref,similaridade,nota,categoria,data,atraso_dias,texto_identico_x,avaliacao
R1,0.952300,5,cama_mesa_banho,2017-12,-1,1,Produto de excelente qualidade. Superou minhas expectativas
R2,0.929700,5,cama_mesa_banho,2017-11,-6,1,bom produto no prazo e embalado
R3,0.934000,5,cama_mesa_banho,2018-04,-36,1,"PRODUTO ÓTIMO, CHEGOU BEM ANTES DA DATA PREVISTA"
R4,0.917300,5,cama_mesa_banho,2018-06,-24,1,Gostei bastante das minhas toalhas são boas
R5,0.924000,5,cama_mesa_banho,2018-01,-16,1,"Chegou antes do prazo, tapete lindo e ótimo"
R6,0.931200,5,cama_mesa_banho,2018-08,-3,1,super recomendado. superou minha expecyativa
R7,0.925500,5,cama_mesa_banho,2018-03,-10,1,"Produto de excelente qualidade, tecido macio e confortável. Recomendo"
R8,0.924300,5,cama_mesa_banho,2016-11,-8,1,"O produto chegou no prazo e conforme,produto de ótima qualidade"
R9,0.928800,5,cama_mesa_banho,2018-07,-6,1,Recebi antes do prazo Material excelente Recomendo
R10,0.924300,5,cama_mesa_banho,2018-03,-9,1,"Produto atendeu minhas expectativas , apesar de ser muito transparente , material muito bom !! Chegou antes do prazo de entrega previsto , super indico!!"


### Temas em experiências de entrega insatisfatórias

In [5]:
r3 = rag.ask("Quais temas aparecem nas avaliações de pedidos que foram entregues com atraso?"); show(r3)

### ❓ Quais temas aparecem nas avaliações de pedidos que foram entregues com atraso?

**Status:** ✅ Respondida com evidências

**Filtro de metadados aplicado:** `{'delivered_late': True}`

**Consultas usadas na busca:** `Quais temas aparecem nas avaliações de pedidos que foram entregues com atraso?` · `o pedido atrasou e chegou muito depois do prazo` · `a entrega demorou mais tempo do que o combinado` · `fiquei decepcionado com a demora na entrega`

**Perfil do tema:** 639 avaliações relacionadas · nota média 1.6 · 81.2% com nota 1-2

**Modelo de geração:** `gemini-3.5-flash-lite`

---

- **Demora excessiva e atraso nas entregas**: Os clientes relatam insatisfação com a lentidão, prazos longos que já existiam e que ainda sofrem atrasos adicionais [R1, R3, R4, R5, R6, R8, R9, R10, R11, R14, R15]. Com base no perfil das avaliações relacionadas, há uma estimativa de 639 avaliações sobre o tema, com nota média de 1.6 e 81,2% das avaliações com nota 1 ou 2.
- **Problemas no transporte e manuseio do produto**: Há relatos de produtos entregues soltos dentro de uma caixa suja, exigindo limpeza da embalagem [R2], além de falhas na rota de entrega com desvios desnecessários entre estados e regiões antes de chegar ao destino final [R12].
- **Impacto negativo na rotina e urgência do consumidor**: O atraso prejudica o andamento de obras [R13], frustra expectativas de quem já estava precisando do item comprado [R15], e gera insatisfação mesmo com o pagamento de frete [R11].


> **Limitações:** A análise é baseada em uma amostra recuperada de 15 avaliações detalhadas e em estimativas globais de um universo de 639 avaliações relacionadas a pedidos com entrega atrasada, podendo conter viés de insatisfação já que 81,2% das avaliações possuem notas 1 ou 2.

**Evidências citadas (14):**

ref,similaridade,nota,categoria,data,atraso_dias,texto_identico_x,avaliacao
R1,0.933000,5,perfumaria,2017-05,1,5,Muita demora na entrega
R2,0.924500,1,esporte_lazer,2018-08,1,1,"Decepcionada com a compra. O produto foi entregue com atraso, o pior não foi isso, o mesmo chegou solto em uma caixa suja, tive que usar ""panos"" para limpar a embalagem antes de abrir. A sujeira me deixou indignada!"
R3,0.925800,2,cool_stuff,2018-02,3,1,Achei q demorou muito a entrega
R4,0.922300,1,relogios_presentes,2017-08,10,1,Pedido já tinha um longo prazo para entrega e ainda atrasa. Produto não recebido ainda.
R5,0.921800,1,utilidades_domesticas,2018-03,13,1,ta demorando a entrega nao resebi minha compra ainda
R6,0.922600,3,esporte_lazer,2018-03,1,1,Demora na entrega e ainda atrasou.
R8,0.932000,1,esporte_lazer,2018-05,3,1,Atrasou na entrega
R9,0.910500,1,beleza_saude,2018-02,15,1,Pessima entrega demora um século pra entregar
R10,0.925100,3,bebes,2018-01,2,1,Demorou demais pra entregar
R11,0.903100,1,perfumaria,2018-08,4,1,Muito Chateado com atraso. Muito chateado em pagar 25 de frete e ao menos enviar no SEDEX 10.. ou algo parecido. esperar quase 20 dias. e nada


### Recorte por período

Maio de 2018 teve a greve nacional dos caminhoneiros. Filtrando o período, é possível investigar se os clientes
relataram efeitos nas entregas — a resposta continua dependendo exclusivamente do que as avaliações dizem.

In [6]:
r4 = rag.ask("O que os clientes relataram sobre a entrega em maio e junho de 2018?"); show(r4)

### ❓ O que os clientes relataram sobre a entrega em maio e junho de 2018?

**Status:** ✅ Respondida com evidências

**Filtro de metadados aplicado:** `{'year_month': {'$in': ['2018-05', '2018-06']}}`

**Consultas usadas na busca:** `O que os clientes relataram sobre a entrega em maio e junho de 2018?` · `o pedido demorou muito para chegar` · `a entrega atrasou bastante` · `recebi o produto depois do prazo combinado` · `o entregador demorou para trazer minha compra`

**Perfil do tema:** 186 avaliações relacionadas · nota média 2.8 · 50.0% com nota 1-2

**Modelo de geração:** `gemini-3.5-flash-lite`

---

Com base na amostra recuperada para o período de maio e junho de 2018, os clientes relataram os seguintes aspectos sobre a entrega: - **Atrasos e lentidão na entrega**: Vários clientes mencionaram atrasos no recebimento [R1, R4, R5, R8], entregas ainda não efetuadas [R5], além de percepções de que a entrega demorou demais [R2] ou que o prazo foi longo [R9]. O perfil das avaliações indica uma nota média de 2.8, com 50% de notas 1 ou 2, e 23.8% de pedidos entregues após o prazo entre os itens entregues. - **Pontos positivos e antecipação**: Houve relatos de clientes que receberam o produto no prazo combinado [R3, R14] ou até mesmo antes do prazo determinado [R7, R15]. - **Erros e incompletude nos pedidos**: Foram reportados problemas como o recebimento de produtos quebrados [R6], entrega de produtos errados [R10] e pedidos que chegaram incompletos, faltando partes dos itens comprados [R11, R12].


> **Limitações:** A análise é baseada em uma amostra de 15 avaliações recuperadas a partir de um total de 186 avaliações semanticamente relacionadas no período de maio e junho de 2018. As frequências e percentuais citados referem-se estritamente a essa amostra e ao perfil recuperado.

**Evidências citadas (14):**

ref,similaridade,nota,categoria,data,atraso_dias,texto_identico_x,avaliacao
R1,0.940900,1,esporte_lazer,2018-05,3.000000,1,Atrasou na entrega
R2,0.934300,4,utilidades_domesticas,2018-06,-5.000000,1,A entrega demorou demais.
R3,0.907900,5,pet_shop,2018-06,-16.000000,1,"Ótimo vendedor, recebi o produto no prazo combinado. Ótimo produto ."
R4,0.922500,2,ferramentas_jardim,2018-05,nan,1,ENTREGA MUITO ATRASADA
R5,0.911400,1,esporte_lazer,2018-06,5.000000,1,Atraso Entrega. A entrega ainda não foi efetuada.
R6,0.911500,1,eletronicos,2018-06,-17.000000,1,Pedido veio quebrado
R7,0.905800,5,beleza_saude,2018-06,-26.000000,1,O produto chegou antes do prazo determinado
R8,0.911300,2,cine_foto,2018-05,5.000000,1,mediana. o produto foi entregue com atraso
R9,0.916200,3,relogios_presentes,2018-06,-17.000000,1,Muita demora. O prazo de entrega foi longo demais
R10,0.913600,1,informatica_acessorios,2018-06,-3.000000,1,Produto entregue errado


### Filtro explícito (uso programático)

Analistas também podem passar o filtro diretamente, sem depender da extração pelo LLM:

In [7]:
r5 = rag.ask("Quais são as principais reclamações dos clientes?",
             where={"$and": [{"product_category": "informatica_acessorios"}, {"review_score": {"$lte": 2}}]})
show(r5)

### ❓ Quais são as principais reclamações dos clientes?

**Status:** ✅ Respondida com evidências

**Filtro de metadados aplicado:** `{'$and': [{'product_category': 'informatica_acessorios'}, {'review_score': {'$lte': 2}}]}`

**Consultas usadas na busca:** `Quais são as principais reclamações dos clientes?` · `o produto veio quebrado e fiquei muito decepcionado` · `a entrega atrasou demais e o atendimento foi péssimo` · `veio faltando peças na minha encomenda` · `o produto não funciona e o vendedor não responde`

**Perfil do tema:** 192 avaliações relacionadas · nota média 1.13 · 100.0% com nota 1-2

**Modelo de geração:** `gemini-3.5-flash-lite`

---

Com base na amostra recuperada de 192 avaliações (todas com notas 1 ou 2), as principais reclamações dos clientes na categoria informatica_acessorios agrupam-se nos seguintes temas: - **Problemas com a entrega e prazos**: Clientes relatam atrasos na entrega, prazos longos estipulados pela loja, demora demasiada [R1, R5, R11], além de pedidos que não foram entregues ou não enviados [R6, R8]. O perfil das avaliações indica que 26.8% dos pedidos entregues sofreram com atraso. - **Produto danificado ou quebrado**: Há relatos de recebimento de mercadorias quebradas, sendo necessária a tentativa de contato sem retorno e solicitação de devolução [R2, R4]. - **Envio de produto errado, divergente ou incompatível**: Clientes apontam o recebimento de itens incorretos, com cores diferentes das solicitadas, modelos errados, mouses com fio enviados no lugar de sem fio (e sem serem gamers conforme o anúncio), além de cartuchos incompatíveis com a impressora [R7, R10, R12, R13, R14]. - **Qualidade e funcionamento do produto**: Ocorre insatisfação com produtos que não funcionam [R3], materiais considerados muito frágeis e finos que não atendem às expectativas [R9], e alegações de recebimento de produtos falsificados com embalagem danificada e falhas de reconhecimento por software [R15].


> **Limitações:** A análise baseia-se em uma amostra de 192 avaliações da categoria informatica_acessorios com notas baixas (1 ou 2), sendo que apenas 15 avaliações detalhadas foram recuperadas no texto. A frequência e a recorrência dos temas refletem exclusivamente esta amostra e as estimativas do perfil fornecido.

**Evidências citadas (15):**

ref,similaridade,nota,categoria,data,atraso_dias,texto_identico_x,avaliacao
R1,0.921300,2,informatica_acessorios,2018-03,0.000000,1,Atraso na entrega.. ruim
R2,0.914100,1,informatica_acessorios,2018-01,-17.000000,1,"Recebi o produto no prazo, porém esta quebrado, já mandei um e-mail e é a segunda avaliação que eu preencho informando o ocorrido. Estou aguardando uma resposta."
R3,0.920900,1,informatica_acessorios,2018-08,-14.000000,2,produto não funciona
R4,0.914500,1,informatica_acessorios,2017-12,-14.000000,1,"o produto veio quebrado ,,, quero devolver"
R5,0.918600,2,informatica_acessorios,2018-08,nan,1,Prazo longo e atraso. A loja pediu quase 1 mês para entregar e ainda atrasou a entrega do produto. E estamos sem receber o produto. . .
R6,0.900700,1,informatica_acessorios,2017-04,nan,1,NAO ENVIOU O MEU PEDIDO
R7,0.908800,1,informatica_acessorios,2018-07,-5.000000,1,Produto errado. Comprei um produto e veio outro. Devolvi o produto.
R8,0.917100,1,informatica_acessorios,2018-07,nan,1,demora e não entrega. péssimo
R9,0.910800,1,informatica_acessorios,2018-08,-10.000000,1,"Nao atendeu minha especta. Produto naõ atemdeu minha espectativa ,material muito fragil e fino"
R10,0.897100,1,informatica_acessorios,2017-10,-13.000000,1,Foi enviado cartucho incompatível com minha impressora.


## 3. Como os metadados melhoram a recuperação

Comparamos a mesma pergunta **com** e **sem** filtros. A métrica é a % das evidências que realmente atendem à
condição pedida na pergunta — sem filtros, a busca semântica só consegue aproximar a condição pelo texto.

In [8]:
casos = [
    ("Quais são os principais problemas relatados nas avaliações com nota baixa?", lambda m: m["review_score"] <= 2),
    ("O que os clientes mais elogiam em produtos de cama, mesa e banho?", lambda m: m["product_category"] == "cama_mesa_banho"),
    ("Quais temas aparecem nas avaliações de pedidos que foram entregues com atraso?", lambda m: m.get("delivered_late") is True),
    ("O que os clientes relataram sobre a entrega em maio e junho de 2018?", lambda m: m["year_month"] in ("2018-05", "2018-06")),
]
rag_sem_filtros = VoCRAG(use_metadata_filters=False)
linhas = []
for q, cond in casos:
    for nome, sistema in [("com filtros", rag), ("sem filtros", rag_sem_filtros)]:
        r = sistema.ask(q)
        evs = r.evidences
        linhas.append({"pergunta": q[:60], "pipeline": nome, "status": r.status, "filtro": str(r.filters_applied or "-"),
                       "evidências": len(evs),
                       "% atende à condição": round(100 * sum(cond(e.metadata) for e in evs) / len(evs)) if evs else None})
comp = pd.DataFrame(linhas)
comp

,pergunta,pipeline,status,filtro,evidências,% atende à condição
0,Quais são os principais problemas relatados na...,com filtros,respondida,{'review_score': {'$lte': 2}},15,100
1,Quais são os principais problemas relatados na...,sem filtros,respondida,-,15,93
2,O que os clientes mais elogiam em produtos de ...,com filtros,respondida,"{'$and': [{'review_score': {'$gte': 4}}, {'pro...",15,100
3,O que os clientes mais elogiam em produtos de ...,sem filtros,respondida,-,15,7
4,Quais temas aparecem nas avaliações de pedidos...,com filtros,respondida,{'delivered_late': True},15,100
5,Quais temas aparecem nas avaliações de pedidos...,sem filtros,respondida,-,15,60
6,O que os clientes relataram sobre a entrega em...,com filtros,respondida,"{'year_month': {'$in': ['2018-05', '2018-06']}}",15,100
7,O que os clientes relataram sobre a entrega em...,sem filtros,evidencias_insuficientes,-,15,0


In [9]:
comp.pivot_table(index="pergunta", columns="pipeline", values="% atende à condição")

pipeline,com filtros,sem filtros
pergunta,,
"O que os clientes mais elogiam em produtos de cama, mesa e b",100.0,7.0
O que os clientes relataram sobre a entrega em maio e junho,100.0,0.0
Quais são os principais problemas relatados nas avaliações c,100.0,93.0
Quais temas aparecem nas avaliações de pedidos que foram ent,100.0,60.0


### Conclusões

* **Precisão contextual:** com filtros, 100% das evidências atendem à condição da pergunta; sem eles, a busca
  semântica traz avaliações sobre o tema, mas de qualquer nota, categoria ou período — e o LLM poderia generalizar
  conclusões a partir de um recorte errado.
* **Perguntas antes impossíveis:** “elogios em uma categoria” ou “reclamações em um mês” não podem ser respondidas
  só pelo texto, pois o cliente raramente escreve a categoria ou a data no comentário.
* **Contexto mais informativo:** o perfil do tema (nota média, % de atraso, categorias) é calculado dentro do recorte.
* **Riscos tratados:** filtros só são aplicados quando pedidos explicitamente, valores são validados contra a base, e
  o limiar de relevância + mínimo de evidências continuam valendo — um recorte muito pequeno resulta em “sem
  evidências”, não em uma resposta inventada.